In [ ]:
# ========================
# 1. IMPORTS
# ========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import lightgbm as lgb
import matplotlib.pyplot as plt

# ========================
# 2. LOAD DATA
# ========================

candidatos = pd.read_csv(
    '/content/Hackaton_Enter_Base_Candidatos.xlsx - Subsídios disponibilizados.csv',
    header=1
)

resultados = pd.read_csv(
    '/content/Hackaton_Enter_Base_Candidatos.xlsx - Resultados dos processos.csv'
)

# ========================
# 3. LIMPEZA DE COLUNAS
# ========================

def limpar_colunas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("ã", "a")
        .str.replace("ç", "c")
        .str.replace("é", "e")
        .str.replace("ê", "e")
        .str.replace("í", "i")
        .str.replace("ó", "o")
        .str.replace("ú", "u")
    )
    return df

candidatos = limpar_colunas(candidatos)
resultados = limpar_colunas(resultados)

# ========================
# 4. PADRONIZAR CHAVE
# ========================

for df_temp in [candidatos, resultados]:
    for col in df_temp.columns:
        if "processo" in col:
            df_temp.rename(columns={col: "numero_do_processo"}, inplace=True)

candidatos["numero_do_processo"] = candidatos["numero_do_processo"].astype(str)
resultados["numero_do_processo"] = resultados["numero_do_processo"].astype(str)

# ========================
# 5. MERGE
# ========================

df = candidatos.merge(resultados, on="numero_do_processo", how="left")
print("Shape:", df.shape)

# ========================
# 6. LIMPAR VALORES MONETÁRIOS
# ========================

def limpar_valor(col):
    return (
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

df["valor_condenacao"] = limpar_valor(df["valor_da_condenacao/indenizacao"])
df["valor_condenacao"] = pd.to_numeric(df["valor_condenacao"], errors='coerce')

# ========================
# 7. REMOVER ACORDO
# ========================

df = df[df["resultado_micro"] != "Acordo"]

# ========================
# 8. FEATURE ENGINEERING
# ========================

cols_docs = [
    'contrato',
    'extrato',
    'comprovante_de_credito',
    'dossie',
    'demonstrativo_de_evolucao_da_divida',
    'laudo_referenciado'
]

df["qtd_docs"] = df[cols_docs].sum(axis=1)

df["doc_score"] = (
    3*df["contrato"] +
    3*df["extrato"] +
    2*df["comprovante_de_credito"] +
    1*df["demonstrativo_de_evolucao_da_divida"]
)

df["combo"] = df["contrato"].astype(str) + "_" + df["extrato"].astype(str)

# ========================
# 9. TARGET BINÁRIO
# ========================

# PERDEU = Procedência + Parcial
df["perdeu"] = df["resultado_micro"].isin([
    "Procedência", "Parcial procedência"
]).astype(int)

# ========================
# 10. FEATURES
# ========================

X = df[cols_docs + ["qtd_docs", "doc_score", "combo", "sub_assunto", "uf"]]
X = pd.get_dummies(X, drop_first=True)

y = df["perdeu"]

# ========================
# 11. TRAIN / TEST
# ========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ========================
# 12. PESOS
# ========================

weights = y_train.map({
    0: 1.0,   # ganhou
    1: 2.0    # perdeu (mais importante)
})

# ========================
# 13. MODELO
# ========================

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    random_state=42
)

model.fit(X_train, y_train, sample_weight=weights)

# ========================
# 14. PREDIÇÃO
# ========================

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

# ========================
# 15. AVALIAÇÃO
# ========================

print("\n===== RESULTADOS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Ganhou", "Perdeu"]))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

# ========================
# 16. FEATURE IMPORTANCE
# ========================

feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

print("\nTop 10 features:")
print(feat_imp.head(10))

plt.figure(figsize=(8,4))
feat_imp.head(10).plot(kind='bar')
plt.title("Top 10 Feature Importance")
plt.show()

# ========================
# 17. DECISÃO (POLÍTICA)
# ========================

df_test = X_test.copy()
df_test["prob_perder"] = y_proba

# regra simples
df_test["decisao"] = np.where(
    df_test["prob_perder"] > 0.6,
    "ACORDO",
    "DEFESA"
)

print("\nExemplos de decisão:")
print(df_test[["prob_perder", "decisao"]].head(10))

# ========================
# 18. EXPECTED LOSS
# ========================

df_test["valor_predito"] = df.loc[X_test.index, "valor_condenacao"]
df_test["expected_loss"] = df_test["prob_perder"] * df_test["valor_predito"]

# decisão final baseada em custo
df_test["decisao_final"] = np.where(
    df_test["expected_loss"] > 3000,
    "ACORDO",
    "DEFESA"
)

print("\nDecisão com custo esperado:")
print(df_test[["prob_perder", "valor_predito", "expected_loss", "decisao_final"]].head(10))

: 

[LightGBM] [Info] Number of positive: 14390, number of negative: 33386
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007154 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 87
[LightGBM] [Info] Number of data points in the train set: 47776, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.462954 -> initscore=-0.148456
[LightGBM] [Info] Start training from score -0.148456

===== MODELO DE RISCO =====
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      8347
           1       0.75      0.83      0.79      3597

    accuracy                           0.87     11944
   macro avg       0.84      0.86      0.85     11944
weighted avg       0.87      0.87      0.87     11944

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.